In [3]:
import os

os.environ[
    "TF_USE_LEGACY_KERAS"
] = "1"

In [5]:
!pip install -q tensorflow-recommenders

   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 108.9/108.9 kB 3.4 MB/s eta 0:00:00


In [6]:
import pandas as pd
import pickle

import tensorflow as tf
import tensorflow_recommenders as tfrs

In [7]:
training_df = pd.read_parquet(
    "/kaggle/input/datasets/selinparmar/train-ds/training_dataset.parquet"
)

with open(
    "/kaggle/input/datasets/selinparmar/feature-vocab/feature_vocab.pkl",
    "rb"
) as file:

    feature_vocab = pickle.load(file)

In [8]:
training_df = (
    training_df
    .sample(
        200000,
        random_state=42
    )
)

print(
    training_df.shape
)

(200000, 16)


In [9]:
training_df[
    'customer_id'
] = (
    training_df[
        'customer_id'
    ].astype(str)
)

training_df[
    'article_id'
] = (
    training_df[
        'article_id'
    ].astype(str)
)

In [10]:
user_vocab = (
    feature_vocab[
        'user_vocab'
    ]
)

favorite_product_vocab = (
    feature_vocab[
        'favorite_product_vocab'
    ]
)

item_vocab = (
    feature_vocab[
        'item_vocab'
    ]
)

product_type_vocab = (
    feature_vocab[
        'product_type_vocab'
    ]
)

season_vocab = (
    feature_vocab[
        'season_vocab'
    ]
)

In [11]:
tf_dataset = (
    tf.data.Dataset
    .from_tensor_slices(
        {
            key:
            training_df[
                key
            ].values

            for key in
            training_df.columns
        }
    )
)

BATCH_SIZE = 2048

train = (
    tf_dataset
    .take(
        int(
            0.8 *
            len(training_df)
        )
    )
    .shuffle(100000)
    .batch(BATCH_SIZE)
    .cache()
)

test = (
    tf_dataset
    .skip(
        int(
            0.8 *
            len(training_df)
        )
    )
    .batch(BATCH_SIZE)
    .cache()
)

I0000 00:00:1779634434.480403      57 gpu_device.cc:2019] Created device /job:localhost/replica:0/task:0/device:GPU:0 with 13757 MB memory:  -> device: 0, name: Tesla T4, pci bus id: 0000:00:04.0, compute capability: 7.5
I0000 00:00:1779634434.486855      57 gpu_device.cc:2019] Created device /job:localhost/replica:0/task:0/device:GPU:1 with 13757 MB memory:  -> device: 1, name: Tesla T4, pci bus id: 0000:00:05.0, compute capability: 7.5


In [12]:
class QueryTower(
    tf.keras.Model
):

    def __init__(self):

        super().__init__()

        # Customer ID embedding
        self.customer_embedding = tf.keras.Sequential([

            tf.keras.layers.StringLookup(
                vocabulary=user_vocab,
                mask_token=None
            ),

            tf.keras.layers.Embedding(
                len(user_vocab) + 1,
                32
            )
        ])

        # Favorite product type embedding
        self.favorite_product_embedding = tf.keras.Sequential([

            tf.keras.layers.StringLookup(
                vocabulary=
                favorite_product_vocab,
                mask_token=None
            ),

            tf.keras.layers.Embedding(
                len(
                    favorite_product_vocab
                ) + 1,
                16
            )
        ])

        # Season embedding
        self.season_embedding = tf.keras.Sequential([

            tf.keras.layers.StringLookup(
                vocabulary=
                season_vocab,
                mask_token=None
            ),

            tf.keras.layers.Embedding(
                len(
                    season_vocab
                ) + 1,
                8
            )
        ])

        # Dense network
        self.dense_layers = tf.keras.Sequential([

            tf.keras.layers.Dense(
                128,
                activation='relu'
            ),

            tf.keras.layers.Dense(64)
        ])

    def call(self, features):

        customer_vector = (
            self.customer_embedding(
                features['customer_id']
            )
        )

        favorite_product_vector = (
            self.favorite_product_embedding(
                features[
                    'favorite_product_type'
                ]
            )
        )

        season_vector = (
            self.season_embedding(
                features['season']
            )
        )

        numeric_features = tf.stack([

            tf.cast(
                features['age'],
                tf.float32
            ),

            tf.cast(
                features[
                    'purchase_frequency'
                ],
                tf.float32
            ),

            tf.cast(
                features[
                    'avg_spend'
                ],
                tf.float32
            ),

            tf.cast(
                features[
                    'repeat_purchase_ratio'
                ],
                tf.float32
            )

        ], axis=1)

        return self.dense_layers(
            tf.concat([

                customer_vector,
                favorite_product_vector,
                season_vector,
                numeric_features

            ], axis=1)
        )

In [13]:
class CandidateTower(
    tf.keras.Model
):

    def __init__(self):

        super().__init__()

        # Article embedding
        self.article_embedding = (
            tf.keras.Sequential([

                tf.keras.layers.StringLookup(
                    vocabulary=item_vocab,
                    mask_token=None
                ),

                tf.keras.layers.Embedding(
                    len(item_vocab) + 1,
                    32
                )
            ])
        )

        # Product type embedding
        self.product_embedding = (
            tf.keras.Sequential([

                tf.keras.layers.StringLookup(
                    vocabulary=
                    product_type_vocab,
                    mask_token=None
                ),

                tf.keras.layers.Embedding(
                    len(
                        product_type_vocab
                    ) + 1,
                    16
                )
            ])
        )

        # Season embedding
        self.season_embedding = (
            tf.keras.Sequential([

                tf.keras.layers.StringLookup(
                    vocabulary=
                    season_vocab,
                    mask_token=None
                ),

                tf.keras.layers.Embedding(
                    len(
                        season_vocab
                    ) + 1,
                    8
                )
            ])
        )

        # Dense network
        self.dense_layers = (
            tf.keras.Sequential([

                tf.keras.layers.Dense(
                    128,
                    activation='relu'
                ),

                tf.keras.layers.Dense(
                    64
                )
            ])
        )

    def call(
        self,
        features
    ):

        article_vector = (
            self.article_embedding(
                features[
                    'article_id'
                ]
            )
        )

        product_vector = (
            self.product_embedding(
                features[
                    'product_type_name'
                ]
            )
        )

        season_vector = (
            self.season_embedding(
                features[
                    'season'
                ]
            )
        )

        numeric_features = tf.stack([

            tf.cast(
                features[
                    'purchase_count'
                ],
                tf.float32
            )

        ], axis=1)

        combined_features = (
            tf.concat([

                article_vector,
                product_vector,
                season_vector,
                numeric_features

            ], axis=1)
        )

        return (
            self.dense_layers(
                combined_features
            )
        )

In [14]:
#candidate dataset
candidate_dataset = (
    tf.data.Dataset
    .from_tensor_slices(
        {
            "article_id":
                training_df[
                    "article_id"
                ],

            "product_type_name":
                training_df[
                    "product_type_name"
                ],

            "season":
                training_df[
                    "season"
                ],

            "purchase_count":
                training_df[
                    "purchase_count"
                ]
                .astype(float)
        }
    )
    .batch(256)
)

In [15]:
class TwoTowerModel(
    tfrs.models.Model
):

    def __init__(self):

        super().__init__()

        self.query_model = (
            QueryTower()
        )

        self.candidate_model = (
            CandidateTower()
        )

        self.task = (
            tfrs.tasks.Retrieval(
                metrics=
                tfrs.metrics.FactorizedTopK(

                    candidates=
                    candidate_dataset.map(
                        self.candidate_model
                    )
                )
            )
        )

    def compute_loss(
        self,
        features,
        training=False
    ):

        query_embeddings = (
            self.query_model(
                features
            )
        )

        candidate_embeddings = (
            self.candidate_model(
                features
            )
        )

        return self.task(
            query_embeddings,
            candidate_embeddings
        )

In [16]:
model = (
    TwoTowerModel()
)

In [17]:
model.compile(

    optimizer=
    tf.keras.optimizers.Adagrad(
        learning_rate=0.1
    )
)

In [18]:
for batch in train.take(1):

    model.compute_loss(
        batch,
        training=False
    )

print("Model built!")

Model built!


In [19]:
history = model.fit(

    train,

    validation_data=test,

    epochs=3
)

Epoch 1/3


I0000 00:00:1779634545.975016     142 service.cc:152] XLA service 0x7c511aad3ed0 initialized for platform CUDA (this does not guarantee that XLA will be used). Devices:
I0000 00:00:1779634545.975071     142 service.cc:160]   StreamExecutor device (0): Tesla T4, Compute Capability 7.5
I0000 00:00:1779634545.975078     142 service.cc:160]   StreamExecutor device (1): Tesla T4, Compute Capability 7.5
I0000 00:00:1779634546.190568     142 cuda_dnn.cc:529] Loaded cuDNN version 91002
I0000 00:00:1779634546.552564     142 device_compiler.h:188] Compiled cluster using XLA!  This line is logged at most once for the lifetime of the process.


79/79 [==============================] - 831s 10s/step - factorized_top_k/top_1_categorical_accuracy: 0.1151 - factorized_top_k/top_5_categorical_accuracy: 0.1158 - factorized_top_k/top_10_categorical_accuracy: 0.1162 - factorized_top_k/top_50_categorical_accuracy: 0.1176 - factorized_top_k/top_100_categorical_accuracy: 0.1181 - loss: 1310715931.8563 - regularization_loss: 0.0000e+00 - total_loss: 1310715931.8563 - val_factorized_top_k/top_1_categorical_accuracy: 0.0000e+00 - val_factorized_top_k/top_5_categorical_accuracy: 0.0000e+00 - val_factorized_top_k/top_10_categorical_accuracy: 2.5000e-05 - val_factorized_top_k/top_50_categorical_accuracy: 1.5000e-04 - val_factorized_top_k/top_100_categorical_accuracy: 2.5000e-04 - val_loss: 4491258.0000 - val_regularization_loss: 0.0000e+00 - val_total_loss: 4491258.0000
Epoch 2/3
79/79 [==============================] - 937s 12s/step - factorized_top_k/top_1_categorical_accuracy: 0.1041 - factorized_top_k/top_5_categorical_accuracy: 0.1066 - 

In [ ]:
#
model.load_weights(
    "/kaggle/input/datasets/selinparmar/two-tower-2/two_tower_weights_2.weights.h5"
)

print(
    "Epoch 2 weights loaded!"
)

In [ ]:
#
history = model.fit(

    train,

    validation_data=test,

    initial_epoch=2,
    epochs=3
)

In [20]:
model.save_weights(
    "/kaggle/working/two_tower_weights.weights.h5"
)

print(
    "Model saved!"
)

Model saved!
